In [ ]:
import numpy as np
from materialite import (
    Material,
    Order4SymmetricTensor,
    Order2SymmetricTensor,
    Box,
    Orientation,
)
from materialite.models.small_strain_fft import SmallStrainFFT, Elastic, LoadSchedule
import logging

logging.basicConfig()

## `SmallStrainFFT` verification

Repeat the example from Willot [1], which consists of a square domain with a stiff inclusion subjected to in-plane shear.

[1] F. Willot, Fourier-based schemes for computing the mechanical response of composites with accurate local fields, Comptes Rendus. Mécanique 343 (2015) 232–245. https://doi.org/10.1016/j.crme.2014.12.005.

Create a `Material` centered at `[0, 0, 0]`

In [ ]:
box = Box(
    min_corner=np.array([-np.inf, 0, -np.inf]),
    max_corner=np.array([0, np.inf, np.inf]),
)
num_points = 512
material = (
    Material(
        [num_points, num_points, 1],
        sizes=[1 - 1 / num_points, 1 - 1 / num_points, 0],
        origin=np.array([-num_points / 2 + 0.5, -num_points / 2 + 0.5, 0]) / num_points,
    )
    .create_uniform_field("phase", 1)
    .create_uniform_field("orientation", Orientation.identity())
    .insert_feature(box, fields={"phase": 2})
)

Visualize the phases in the material. The red region (`phase = 2`) represents the stiff inclusion.

In [ ]:
material.plot("phase")

Set up the boundary conditions: the mean strain tensor is fully specified, with all components equal to zero except $E_{xy}$

In [ ]:
strain_rate = Order2SymmetricTensor.from_strain_voigt(np.array([0, 0, 0, 0, 0, 1.0]))
stress_rate = Order2SymmetricTensor.zero()
stress_mask = np.zeros(6)
load_schedule = LoadSchedule.from_constant_rates(
    strain_rate=strain_rate, stress_rate=stress_rate, stress_mask=stress_mask
)

Set up the constitutive models. Both phases are isotropic elastic, with phase 2 having a $1000\times$ greater stiffness.

In [ ]:
E1 = 200
ratio = 1000
E2 = E1 * ratio
nu = 0.25
stiffness1 = Order4SymmetricTensor.from_isotropic_constants(
    modulus=E1, shear_modulus=E1 / (2 * (1 + nu))
)
stiffness2 = Order4SymmetricTensor.from_isotropic_constants(
    modulus=E2, shear_modulus=E2 / (2 * (1 + nu))
)
elastic1 = Elastic(stiffness=stiffness1)
elastic2 = Elastic(stiffness=stiffness2)
material = material.create_regional_fields(
    region_label="phase",
    regional_fields={"phase": [1, 2], "constitutive_model": [elastic1, elastic2]},
)

Run one load step of the model.

In [ ]:
model = SmallStrainFFT(load_schedule=load_schedule, end_time=0.001)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
material = model.run(
    material,
    phase_label="phase",
    global_tolerance=1.0,
    strain_correction_tolerance=1.0,
    linear_solver_tolerance=1.0e-9,
)
logger.setLevel(logging.WARNING)

Plot the shear stress near the boundary between the inclusin and the matrix. Compare with the bottom-left plot in Fig. 3 in [1]. Note that the paper did not provide unnormalized stiffness magnitudes, so the plot cannot be reproduced identically.

In [ ]:
lb = 0.1
ub = 3.5 / 1.5 * lb
material.crop(x=[-0.04, 0.04], y=[-0.04, 0.04]).plot(
    "stress", component=5, color_lims=[lb, ub], colormap="jet"
)